In [1]:
import pandas as pd
import numpy as np
import torch

%load_ext autoreload
%autoreload 1
%aimport deep_linear_bandits.data

In [2]:
krb_train, krb_val = deep_linear_bandits.data.preprocess_krbig_interactions()

krb_train, krb_val

(          user_id  video_id
 2826486      1609      2789
 12246800     7003      4363
 11013353     6298      7631
 4759694      2747      8794
 542259        299      3770
 ...           ...       ...
 11166663     6384      2264
 11627800     6658      4374
 11628117     6658     10653
 11628144     6658      3400
 11628147     6658      5047
 
 [677144 rows x 2 columns],
           user_id  video_id
 11005224     6294      5439
 7831171      4452      1904
 6764137      3858      3995
 4223186      2424      6811
 2132262      1215      5865
 ...           ...       ...
 10341213     5901      1064
 2012186      1146      1936
 10877374     6216      4762
 11718768     6713      7018
 5876034      3366      2771
 
 [169270 rows x 2 columns])

In [13]:
ic =deep_linear_bandits.data.preprocess_item_categories()

ic

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [4]:
uf_cat, uf_num = deep_linear_bandits.data.preprocess_user_features()

uf_cat, uf_num

((array([[0, 0, 0, ..., 0, 0, 4],
         [0, 0, 0, ..., 1, 2, 1],
         [0, 0, 0, ..., 0, 0, 6],
         ...,
         [0, 0, 0, ..., 1, 2, 6],
         [0, 0, 0, ..., 0, 0, 1],
         [0, 0, 1, ..., 2, 4, 6]], shape=(7176, 26)),
  [2,
   2,
   2,
   2,
   8,
   30,
   1076,
   13,
   10,
   3,
   47,
   340,
   7,
   5,
   3,
   3,
   3,
   3,
   3,
   3,
   3,
   4,
   8,
   7,
   7,
   7]),
 array([[-1.0639045 , -1.0757797 , -0.6165203 , -0.799442  ],
        [ 1.2634817 ,  0.31302404,  0.5586607 ,  0.51872677],
        [-0.20345363, -1.0757797 , -0.6165203 , -0.7044635 ],
        ...,
        [ 1.5231189 ,  0.12047071,  0.5586607 , -0.27516514],
        [ 1.7709502 , -1.0757797 , -0.6165203 ,  0.15791167],
        [ 0.5019767 ,  2.0164826 ,  3.1556127 , -0.27516514]],
       shape=(7176, 4), dtype=float32))

In [5]:
from torch.utils.data import DataLoader

t = deep_linear_bandits.data.KRBig(
    krb_train,
    uf_cat[0],
    uf_num,
    ic
)

d = DataLoader(
    dataset=t,
    batch_size=1024,
    shuffle=True
)

next(iter(d))

{'user_id': tensor([7146, 1958, 3290,  ..., 4257, 2745, 2930]),
 'item_id': tensor([5884, 2894, 8830,  ...,  767,  961, 9645]),
 'user_cat_feats': tensor([[0, 0, 0,  ..., 0, 0, 1],
         [0, 0, 1,  ..., 2, 0, 6],
         [0, 0, 0,  ..., 1, 0, 1],
         ...,
         [0, 0, 1,  ..., 1, 2, 5],
         [0, 0, 0,  ..., 0, 0, 6],
         [0, 0, 1,  ..., 2, 0, 5]]),
 'user_numeric_feats': tensor([[-1.2904, -1.0758, -0.6165, -0.0506],
         [ 0.7151,  1.5093, -0.6165, -0.6165],
         [ 1.0827, -0.4777, -0.6165,  0.6193],
         ...,
         [-0.3133, -0.4777,  0.1249,  1.9749],
         [-0.4822, -1.0758, -0.6165, -0.5524],
         [ 0.6627,  1.3690, -0.6165,  2.7415]]),
 'item_categories': tensor([[0., 1., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.]])}

In [6]:
%aimport deep_linear_bandits.two_tower

In [7]:
model = deep_linear_bandits.two_tower.UserTower(
    uf_cat[1],
    [8] * len(uf_cat[1]),
    uf_num.shape[1]
)

In [8]:
b = next(iter(d))

model(
    b['user_id'],
    b['user_cat_feats'],
    b['user_numeric_feats']
)

tensor([[ 0.2278,  0.0078,  0.0155,  ...,  0.1614, -0.1121,  0.0194],
        [ 0.0695,  0.1321,  0.1275,  ..., -0.0584, -0.0537, -0.1290],
        [ 0.1690, -0.0290,  0.1561,  ...,  0.1916,  0.0109, -0.1546],
        ...,
        [ 0.2232,  0.0638,  0.1031,  ..., -0.0245, -0.1765, -0.1491],
        [ 0.1539,  0.1782, -0.0285,  ...,  0.0997, -0.0775,  0.0579],
        [ 0.1271, -0.1468,  0.1430,  ..., -0.1234, -0.2287, -0.1827]],
       grad_fn=<DivBackward0>)

In [9]:
b["item_id"], b["item_categories"]

(tensor([1254, 6746,  576,  ..., 8814,  148, 7521]),
 tensor([[0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 1., 0., 0.]]))

In [10]:
im = deep_linear_bandits.two_tower.ItemTower(
    ic.shape[1]
)

im(
    b["item_id"],
    b["item_categories"]
)

tensor([[ 0.1048, -0.1299, -0.1416,  ..., -0.2657,  0.2015,  0.0091],
        [ 0.1110, -0.3232,  0.0090,  ..., -0.0601, -0.0115, -0.1147],
        [ 0.0906, -0.0881, -0.1206,  ...,  0.1524,  0.1214, -0.0981],
        ...,
        [ 0.1690, -0.0295, -0.0010,  ...,  0.0051, -0.0604, -0.0853],
        [ 0.0140, -0.1205, -0.1816,  ..., -0.0641,  0.0466,  0.0678],
        [ 0.0215, -0.0071, -0.2369,  ..., -0.1495,  0.1293,  0.1011]],
       grad_fn=<DivBackward0>)

In [11]:
tt = deep_linear_bandits.two_tower.TwoTower(
    uf_cat[1],
    [8] * len(uf_cat[1]),
    uf_num.shape[1],
    ic.shape[1]
)

In [17]:
ic, ic.shape

(tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 torch.Size([10728, 31]))

In [25]:
rnd = torch.randint(0, 10728, (256,))

rnd

tensor([ 5608,  7171,  9531,  5547,  7537,  3300,  9569,  5738,  4542, 10521,
         9130,  1386,  6074,  7050,  6998,  8416,  9509,  9786,  3338,  8745,
        10512,  7280,  6173,  6194,  4408,  7341,  5394,  7402,  2815,   414,
         9102,  6585,  8731,  2673,  2422,  6833,  4202,    82,  8604,   102,
         4702,  5784,  4867,  1985,  4571,  3815,  1816,  9643,  7483,  2055,
         8325,  5177,  2859,  6183,  2652,  6699,  8378,  9953, 10292,  7964,
         5606,  4273,  1880,  8822,  4897,  9921,   305,  6100,  7610, 10659,
         8742,  6347,  8849,   468,  2546, 10181,   761,  3165,   782,  6058,
         6889,   453,  8865,  7660,  5676,  5509,   597,  5443,  8866,  9470,
          598,  3347,  3246, 10609, 10531,  2488,  4739,  6582,  5251,  2341,
         1923,   920,  3550,  9896,  4869,  9197,   366,   894,  9188,   909,
         7422,  2655,  3426,  7668,  5212,  3079,  9432,  9375, 10223, 10508,
         3169,  2196,  8098,  5771,  4504,  3616,   415,  6673, 

In [37]:
ic[rnd]

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])